# Testing the per round error rate for Tesseract and relay BP: Arianna Wrappers

In [1]:
# get the parity check matrix / logicals for the BB codes
from realtime_decoding.decoder_switching import get_BB_circuit
from quits.qldpc_code import BbCode
from quits import ErrorModel

def get_bb_code_parity_and_logs(d):

    d_dict = {6:{'l':6, 'm':6, 'A_x_pows': [3], 'A_y_pows': [1,2], 'B_x_pows': [1,2], 'B_y_pows':[3]},
                10: {'l':15, 'm':3, 'A_x_pows': [9], 'A_y_pows': [1,2], 'B_x_pows': [2,7], 'B_y_pows':[0]},
                12:{'l':12, 'm':6, 'A_x_pows': [3], 'A_y_pows': [1,2], 'B_x_pows': [1,2], 'B_y_pows':[3]},}
    code_params = d_dict[d]
    bb = BbCode(
        l=code_params['l'],
        m=code_params['m'],
        A_x_pows=code_params['A_x_pows'],
        A_y_pows=code_params['A_y_pows'],
        B_x_pows=code_params['B_x_pows'],
        B_y_pows=code_params['B_y_pows'],
    )
    return bb.hx, bb.lx

In [2]:
# get UF thresholds... then add them to the plot
from realtime_decoding.decoder_switching import get_BB_circuit, get_rsc_circuits
from realtime_decoding.decoding import UnionFindWrapper, TesseractWrapper, RelayBpWrapper  
from quits import sliding_window_circuit_mem
from filelock import FileLock
import numpy as np
import pandas as pd
import psutil
import os


# decoder_params = {"window_observables":[],
#                 #   "priors":[],
#                   'det_beam': 10,
#                   'pqlimit': 50_000,
#                   'beam_climbing': True,
#                   'no_revisit_dets': True}

decoder_params =  {
        'gamma_dist_interval': (-0.175, 0.575),
        'gamma0': 0.125,
        'pre_iter': 80,
        'num_sets': 300,
        'set_max_iter': 60,
        'stop_nconv' : 1 }

def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)  # Return in MB


def run_single_trial(p, d, code_type, num_shots, rounds,basis, csv_filename):
    task_id = f"p={p:.4f}, d={d}"
    print(f"[{task_id}] Starting trial. Memory: {get_memory_usage():.2f} MB")
    nbuffer = d            #W-F = buffer -> W = buffer +F
    F       = d//2         #just pick this commit region so that there are at least some windows to decode for rds>20 (e.g. for d=12, we have 20 rounds, and W = 12 +6 = 18)
    W       = nbuffer + F  #entire window

    # 1. Circuit Generation
    if code_type == "RSC":
        rsc_circuits, rsc_codes = get_rsc_circuits(p, [d], basis)
        circuit, code_params = rsc_circuits[0], rsc_codes[0]
    else:
        bb_circuit, bb_code = get_BB_circuit(d, basis, p, rounds)
        circuit = bb_circuit
        code_params = (bb_code.hx, bb_code.lx) if basis == 'x' else (bb_code.hz, bb_code.lz)

    sampler = circuit.compile_detector_sampler()
    det_events, obs_flips = sampler.sample(shots=num_shots, separate_observables=True)

    # 2. Decoding using TesseractWrapper
    logical_pred = sliding_window_circuit_mem(
        det_events, circuit, code_params[0], code_params[1],
        W, F, 
        RelayBpWrapper, RelayBpWrapper, 
        decoder_params, decoder_params,
        'priors', 'priors', 'decode', 'decode'
    )    
    pL = np.mean((obs_flips - logical_pred).any(axis=1))
    eps = 1-(1-pL)**(1/rounds)

    # 3. Log Preparation
    row = {
        'cutoff':0.0,'p': p, 'd': d, 'basis': basis,
        'num_shots': num_shots, 'code_type': code_type, 'cluster_metric':"size",
        'rounds': rounds, 'eps': eps
    }
    
    lock_path = csv_filename + ".lock"
    with FileLock(lock_path):
        df_row = pd.DataFrame([row])
        header_needed = not os.path.exists(csv_filename)
        df_row.to_csv(csv_filename, mode='a', index=False, header=header_needed)
    print(f"Logged results to {csv_filename}: {row}")
    print(f"[{task_id}] Finished trial. Memory: {get_memory_usage():.2f} MB")
    return row



In [ ]:
from joblib import Parallel, delayed


p_list = np.logspace(-3.5,-2.5,5)
d_list = [6, 10, 12]
num_shots = 10_000
code_type = "BB"
rounds = 22
csv_file = "/Users/ariannameinking/Documents/Brown_Research/realtime_decoding_qldpc/data/raw/tesseract_characterization.csv"

tasks = [
    (p, d, code_type, num_shots, rounds, 'x', csv_file)
    for p in p_list 
    for d in d_list 
]

# 4. Parallel Execution
print(f"Starting {len(tasks)} parallel tasks...")
results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_single_trial)(*task) for task in tasks
)

print("All simulation tasks completed.")

Starting 15 parallel tasks...
[p=0.0006, d=6] Starting trial. Memory: 213.33 MB
[p=0.0003, d=12] Starting trial. Memory: 212.34 MB
[p=0.0018, d=6] Starting trial. Memory: 213.67 MB
[p=0.0010, d=10] Starting trial. Memory: 214.69 MB
[p=0.0006, d=12] Starting trial. Memory: 214.66 MB
[p=0.0006, d=10] Starting trial. Memory: 213.97 MB
[p=0.0003, d=10] Starting trial. Memory: 213.50 MB
[p=0.0010, d=12] Starting trial. Memory: 213.45 MB
[p=0.0010, d=6] Starting trial. Memory: 213.36 MB
[p=0.0003, d=6] Starting trial. Memory: 214.47 MB
Logged results to /Users/ariannameinking/Documents/Brown_Research/realtime_decoding_qldpc/data/raw/tesseract_characterization.csv: {'cutoff': 0.0, 'p': np.float64(0.00031622776601683794), 'd': 6, 'basis': 'x', 'num_shots': 10000, 'code_type': 'BB', 'cluster_metric': 'size', 'rounds': 22, 'eps': np.float64(0.0)}
[p=0.0003, d=6] Finished trial. Memory: 245.95 MB
[p=0.0018, d=10] Starting trial. Memory: 245.97 MB
Logged results to /Users/ariannameinking/Documents